In [1]:
import os, sqlite3

In [2]:
DB_PATH = "experiment.db"
if os.path.exists(DB_PATH):
    os.remove(DB_PATH)

def rows(conn):
    return conn.execute("SELECT id, name FROM clients ORDER BY id").fetchall()

conn = sqlite3.connect(DB_PATH, autocommit=False, timeout=1)
conn.execute("""
    CREATE TABLE clients (
        id INTEGER PRIMARY KEY AUTOINCREMENT,
        name TEXT NOT NULL,
        CONSTRAINT uq_clients_name UNIQUE (name)
    )
""")

conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm A",))
conn.commit()

observer = sqlite3.connect(DB_PATH, autocommit=False, timeout=1)

def look(label):
    print(f"{label:<28} writer: {rows(conn)}")
    print(f"{'':<28} observer: {rows(observer)}")
    observer.rollback()

look("start")

start                        writer: [(1, 'Farm A')]
                             observer: [(1, 'Farm A')]


In [3]:
conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm B",))
look("after valid insert")

try:
    conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm A",))
except sqlite3.Error as err:
    print("type:", type(err).__name__)
    print("message: ", err)
    print("sqlite_errorname: ", getattr(err, "sqlite_errorname", None))
    print("sqlite_errorcode: ", getattr(err, "sqlite_errorcode", None))

print("in_transaction: ", conn.in_transaction)
look("after failed duplicate")

after valid insert           writer: [(1, 'Farm A'), (2, 'Farm B')]
                             observer: [(1, 'Farm A')]
type: IntegrityError
message:  UNIQUE constraint failed: clients.name
sqlite_errorname:  SQLITE_CONSTRAINT_UNIQUE
sqlite_errorcode:  2067
in_transaction:  True
after failed duplicate       writer: [(1, 'Farm A'), (2, 'Farm B')]
                             observer: [(1, 'Farm A')]


In [4]:
conn.execute("INSERT INTO clients (name) VALUES (?)", ("Farm C",))
look("after another valid insert")

after another valid insert   writer: [(1, 'Farm A'), (2, 'Farm B'), (3, 'Farm C')]
                             observer: [(1, 'Farm A')]


In [5]:
try:
    observer.execute("INSERT INTO clients (name) VALUES (?)", ("Farm Z",))
    print("observer write without commit succeeded")
    observer.commit()
    print("observer write succeeded")
except sqlite3.Error as err:
    print("observer write failed: ", type(err).__name__, " - ", err)
    observer.rollback()

observer write failed:  OperationalError  -  database is locked


In [6]:
conn.commit()
look("after commit")

after commit                 writer: [(1, 'Farm A'), (2, 'Farm B'), (3, 'Farm C')]
                             observer: [(1, 'Farm A'), (2, 'Farm B'), (3, 'Farm C')]


In [7]:
target_id = conn.execute("SELECT id FROM clients WHERE name = ?", ("Farm C",)).fetchone()[0]
try:
    conn.execute("UPDATE clients SET name = ? WHERE id = ?", ("Farm A", target_id))
except sqlite3.Error as err:
    print("type: ", type(err).__name__ , 
          " | message: ", err, 
          " | errorname: ", getattr(err, "sqlite_errorname", None), 
          " | errorcode: ", getattr(err, "sqlite_errorcode", None))
conn.rollback()
look("after failed UPDATE + rollback")

type:  IntegrityError  | message:  UNIQUE constraint failed: clients.name  | errorname:  SQLITE_CONSTRAINT_UNIQUE  | errorcode:  2067
after failed UPDATE + rollback writer: [(1, 'Farm A'), (2, 'Farm B'), (3, 'Farm C')]
                             observer: [(1, 'Farm A'), (2, 'Farm B'), (3, 'Farm C')]


In [8]:
conn.close()
observer.close()